# Train PPO Agent on Ms. Pac-Man

This notebook demonstrates training a Proximal Policy Optimization (PPO) agent.

In [ ]:
# For Google Colab
# !git clone https://github.com/SABRYOLA/pacman.git
# %cd pacman
# !pip install -r requirements.txt

In [ ]:
import torch
import yaml
import sys
sys.path.append('..')

from src.agents import PPOAgent
from src.environment import create_vec_env, create_env
from src.training import OnPolicyTrainer
from src.utils import Logger, CheckpointManager
from src.networks import get_device

In [ ]:
# Load and modify config
with open('../configs/ppo_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['total_timesteps'] = 100000  # Quick demo
config['n_envs'] = 4  # Reduce for notebook

print('PPO Configuration:')
for k, v in config.items():
    print(f'  {k}: {v}')

In [ ]:
device = get_device(config['device'])
print(f'Using device: {device}')

env = create_vec_env(config['env_name'], n_envs=config['n_envs'], seed=config['seed'])
eval_env = create_env(config['env_name'], seed=config['seed'] + 1000)
n_actions = env.single_action_space.n
print(f'Environment: {config["env_name"]}, Actions: {n_actions}')

In [ ]:
agent = PPOAgent(
    n_actions=n_actions,
    learning_rate=config['learning_rate'],
    gamma=config['gamma'],
    gae_lambda=config['gae_lambda'],
    clip_range=config['clip_range'],
    n_epochs=config['n_epochs'],
    n_steps=config['n_steps'],
    n_envs=config['n_envs'],
    batch_size=config['batch_size'],
    ent_coef=config['ent_coef'],
    vf_coef=config['vf_coef'],
    max_grad_norm=config['max_grad_norm'],
    device=device
)

print(f'Agent created with {sum(p.numel() for p in agent.policy.parameters()):,} parameters')

In [ ]:
logger = Logger('../logs/ppo_notebook', 'PPO')
checkpoint_manager = CheckpointManager('../models', 'PPO')

trainer = OnPolicyTrainer(
    agent=agent,
    env=env,
    logger=logger,
    checkpoint_manager=checkpoint_manager,
    total_timesteps=config['total_timesteps'],
    eval_env=eval_env,
    eval_freq=config['eval_freq'],
    eval_episodes=config['eval_episodes']
)

print('Trainer ready!')

In [ ]:
print(f"Starting training for {config['total_timesteps']:,} steps...\n")
trainer.train()
print('Training complete!')

In [ ]:
stats = logger.get_stats()
print('Training Statistics:')
for k, v in stats.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.2f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../logs/

In [ ]:
agent.save('../models/ppo_notebook_final.pt')
env.close()
eval_env.close()
logger.close()
print('Done!')